# Book Video Studio — direct Colab engine
Run lightweight setup, then direct engine setup, then the studio interface. Select GPU when Colab grants access, or explicitly choose experimental CPU. Existing Drive project and clips are preserved.
Hosted Hugging Face inference is removed. The open model and Python libraries still download from Hugging Face/PyPI. No ZeroGPU account or token required. This is not a completely Hugging Face-free model distribution.
Direct output is a small five-second draft (41 frames at 8 fps); quality and speed differ from the hosted demo. Voices, recordings, sound effects, reference uploads and saved project memory remain. Speech beyond the footage holds the last frame; exact lip sync is not included.


In [ ]:
from pathlib import Path
import sys,subprocess
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive')
subprocess.run([sys.executable,'-m','pip','install','--quiet','ipywidgets','Pillow','psutil'],check=True)
print('Studio interface ready. Run the direct engine setup before generating.')


## Direct engine setup
Run once per fresh Colab runtime. Model files download on first generation; reserve at least 20 GB free disk and 10 GB available RAM. CPU generation may be impractically slow.


In [ ]:
from pathlib import Path
import subprocess,sys,shutil,os
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive')
root=Path('/content/drive/MyDrive/C-H-E-N 3D/LTX_Book_Studio');root.mkdir(parents=True,exist_ok=True)
env=Path('/content/ltx_cpu_env');python=env/'bin/python'
log=root/'setup.log'
def setup_run(args):
    result=subprocess.run(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    with log.open('a') as stream:stream.write(result.stdout+'\n')
    print('\n'.join(result.stdout.splitlines()[-15:]))
    if result.returncode:raise RuntimeError('Setup failed. Full output saved to '+str(log))
# Colab may omit ensurepip/python-venv. virtualenv supplies pip independently.
healthy=python.is_file() and subprocess.run([str(python),'-m','pip','--version'],stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL).returncode==0
if not healthy:
    setup_run([sys.executable,'-m','pip','install','--quiet','virtualenv'])
    setup_run([sys.executable,'-m','virtualenv','--clear','--system-site-packages',str(env)])
setup_run([str(python),'-m','pip','install','diffusers==0.36.0','transformers==4.57.1','accelerate==1.11.0','huggingface-hub==0.36.0','sentencepiece','imageio','imageio-ffmpeg','safetensors','psutil','Pillow'])
setup_run([str(python),'-c',"import torch; from diffusers import LTXPipeline,LTXImageToVideoPipeline; from transformers import T5EncoderModel; print('CPU environment ready; torch',torch.__version__)"])
print('Setup ready. Free disk GB:',round(shutil.disk_usage('/content').free/1024**3,1))


In [ ]:
from pathlib import Path
import json,os,time,uuid,subprocess,shutil,psutil,hashlib,urllib.request,zipfile
from datetime import datetime,timezone
import ipywidgets as w
from IPython.display import display,Video,clear_output
project=Path('/content/drive/MyDrive/C-H-E-N 3D/LTX_Book_Studio');project.mkdir(parents=True,exist_ok=True)
record=project/'project.json'
def save(data):
    temporary=record.with_suffix('.tmp');temporary.write_text(json.dumps(data,indent=2));temporary.replace(record)
def load():
    return json.loads(record.read_text()) if record.is_file() else {'schema_version':1,'book':'The War Beneath','characters':{name:'' for name in ['CHEN','COLE','VASIK','OSEI','ASANTE','TORRES','WEBB']},'world':'Meridian exploration ship. Copper-lit KR-9.','style':'Cinematic realistic science fiction. Practical cockpit lighting.','seed':42,'reference_image':'','clips':[]}
data=load()
backend=w.Dropdown(options=['Direct Colab GPU','Experimental CPU'],value='Direct Colab GPU',description='Engine:')
upload=w.Button(description='Upload reference')
backup=w.Button(description='Export project backup')
assemble=w.Button(description='Assemble saved clips')
prompt=w.Textarea(value=data.get('last_prompt',''),description='Scene:',placeholder='Describe action, framing and camera movement...',layout=w.Layout(width='95%',height='110px'))
cast=w.Text(value='',description='Cast:',placeholder='CHEN, COLE (leave blank for an empty cockpit)',layout=w.Layout(width='95%'))
characters=w.Textarea(value=json.dumps(data['characters'],indent=2),description='Characters:',layout=w.Layout(width='95%',height='180px'))
world=w.Textarea(value=data['world'],description='Locations:',layout=w.Layout(width='95%',height='70px'))
style=w.Text(value=data['style'],description='Style:',layout=w.Layout(width='95%'))
reference=w.Text(value=data.get('reference_image',''),description='Image path:',placeholder='Optional full Drive path to an approved reference image',layout=w.Layout(width='95%'))
seed=w.IntText(value=data['seed'],description='Seed:')
steps=w.Dropdown(options=[8,16,24,50],value=24,description='Steps:')
duration_choice=w.Dropdown(options=[5,8,10],value=data.get('motion_seconds',5),description='Motion (s):')
def motion_frames(seconds,fps=8):
    import math
    if seconds not in (5,8,10):raise ValueError('Choose 5, 8 or 10 seconds.')
    return 8*math.ceil(seconds*fps/8)+1
size=w.Dropdown(options=['256×160','192×128'],value='192×128',description='Size:')
savebutton=w.Button(description='Save project')
button=w.Button(description='Generate video',button_style='primary')
approve=w.Button(description='Approve last frame')
history=w.Button(description='Show clip history')
output=w.Output()
def persist():
    parsed=json.loads(characters.value)
    if not isinstance(parsed,dict) or any(not isinstance(v,str) for v in parsed.values()):raise ValueError('Characters must be a JSON object mapping names to descriptions.')
    current=load();current.update(voices=json.loads(voice_map.value),last_speech=speech.value,characters=parsed,world=world.value,style=style.value,seed=seed.value,reference_image=reference.value,last_prompt=prompt.value,motion_seconds=duration_choice.value);save(current);return current
def save_click(_):
    with output:
        try:persist();print('Project saved. It will reload when you reopen this notebook.')
        except Exception as exc:print('Save failed:',exc)
def show_history(_):
    with output:
        for clip in load()['clips'][-20:]:print(clip['id'],clip['status'],clip['prompt'],clip['folder'])
def approve_click(_):
    with output:
        current=load();success=[c for c in current['clips'] if c['status']=='complete']
        if not success:print('No completed clip to approve.');return
        frame=Path(success[-1]['folder'])/'last_frame.png'
        if not frame.is_file():print('Last frame is missing.');return
        approved=project/'approved_references';approved.mkdir(exist_ok=True)
        target=approved/(success[-1]['id']+'.png');shutil.copy2(frame,target)
        current['reference_image']=str(target);save(current);reference.value=str(target)
        print('Approved image saved. Future generation will use it as the starting frame, unless you clear Image path.')

encode="import os\nos.environ['USE_TF']='0';os.environ['USE_FLAX']='0'\nimport torch\nfrom transformers import T5EncoderModel,AutoTokenizer\nfrom pathlib import Path\ntorch.set_num_threads(2)\nimport json\nroot=Path(os.environ['LTX_RUN_DIR'])\nconfig=json.loads((root/'request.json').read_text())\nrepo='Lightricks/LTX-Video'\nprompt=config['effective_prompt']\nprint('Loading tokenizer and text encoder on CPU',flush=True)\ntok=AutoTokenizer.from_pretrained(repo,subfolder='tokenizer')\nmodel=T5EncoderModel.from_pretrained(repo,subfolder='text_encoder',torch_dtype=torch.bfloat16,low_cpu_mem_usage=True).eval()\n# Process one prompt at a time; avoid doubling text-encoder activation memory.\ntexts=[('prompt',prompt)]\nif config['guidance']>1:texts.append(('negative_prompt','blurry, distorted, flickering, text, watermark'))\nencoded={}\nfor label,text in texts:\n    count=len(tok(text).input_ids)\n    if count>config['max_tokens']:print('PROMPT_TRUNCATION',label,count,'tokens; limit',config['max_tokens'],flush=True)\n    inputs=tok(text,padding='max_length',max_length=config['max_tokens'],truncation=True,return_tensors='pt')\n    with torch.inference_mode(): embedding=model(input_ids=inputs.input_ids,attention_mask=inputs.attention_mask).last_hidden_state\n    encoded[label+'_embeds']=embedding\n    encoded[label+'_attention_mask']=inputs.attention_mask\n    del inputs,embedding\ntorch.save(encoded,root/'embeddings.pt')\nprint('TEXT_ENCODING_COMPLETE',flush=True)\n"
generate="import os\nos.environ['USE_TF']='0';os.environ['USE_FLAX']='0'\nimport torch,time\nfrom pathlib import Path\nfrom diffusers import LTXPipeline,LTXImageToVideoPipeline\nfrom diffusers.utils import export_to_video,load_image\ntorch.set_num_threads(2)\nimport json\nroot=Path(os.environ['LTX_RUN_DIR'])\nconfig=json.loads((root/'request.json').read_text())\ndevice=config.get('device','cpu')\nif device=='cuda' and not torch.cuda.is_available():raise RuntimeError('GPU unavailable')\ndtype=(torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if device=='cuda' else torch.bfloat16\nprint('Generation dtype:',dtype,flush=True)\nprint('Loading video components for',device,flush=True)\nPipeline=LTXImageToVideoPipeline if config['reference_image'] else LTXPipeline\npipe=Pipeline.from_pretrained('Lightricks/LTX-Video',text_encoder=None,tokenizer=None,torch_dtype=dtype,low_cpu_mem_usage=True)\nif device=='cuda':pipe.enable_model_cpu_offload()\nelse:pipe.to('cpu')\npipe.vae.enable_tiling()\nemb=torch.load(root/'embeddings.pt',map_location='cpu',weights_only=True)\nemb={k:(v.to(dtype=dtype) if v.is_floating_point() else v) for k,v in emb.items()}\nstart=time.time()\ndef progress(pipe,step,timestep,kwargs):\n    elapsed=time.time()-start\n    remaining=elapsed/(step+1)*(config['steps']-step-1)\n    print('STEP',step+1,'elapsed minutes',round(elapsed/60,1),'estimated remaining minutes',round(remaining/60,1),flush=True)\n    return kwargs\nfrom PIL import ImageOps\nextra={'image':ImageOps.fit(load_image(config['reference_image']).convert('RGB'),(config['width'],config['height']))} if config['reference_image'] else {}\nwith torch.inference_mode():\n    frames=pipe(**extra,**emb,width=config['width'],height=config['height'],num_frames=config['frames'],frame_rate=config['fps'],num_inference_steps=config['steps'],guidance_scale=config['guidance'],generator=torch.Generator(device='cpu').manual_seed(config['seed']),callback_on_step_end=progress,decode_timestep=.05,decode_noise_scale=.025).frames[0]\nframes[-1].save(root/'last_frame.png')\nexport_to_video(frames,str(root/'ltx_cpu_test.mp4'),fps=config['fps'])\nprint('VIDEO_READY',root/'ltx_cpu_test.mp4',flush=True)\n"

def upload_click(_):
    with output:
        from google.colab import files
        from PIL import Image
        selected=files.upload()
        assets=project/'references';assets.mkdir(exist_ok=True)
        for name,content in selected.items():
            import io
            image=Image.open(io.BytesIO(content));image.verify()
            target=assets/(uuid.uuid4().hex[:8]+'.png')
            Image.open(io.BytesIO(content)).convert('RGB').save(target)
            reference.value=str(target);persist();print('Saved reference:',target)
def backup_click(_):
    with output:
        persist();target=project.parent/'Book_Video_Studio_Backup.zip'
        with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED) as archive:
            for path in project.rglob('*'):
                if path.is_file() and 'embedding_cache' not in path.parts:archive.write(path,path.relative_to(project))
        from google.colab import files
        files.download(str(target))

def assemble_click(_):
    with output:
        try:
            current=load();clips=[Path(c.get('voiced_video') or str(Path(c['folder'])/'ltx_cpu_test.mp4')) for c in current['clips'] if c['status']=='complete']
            clips=[p for p in clips if p.is_file()]
            if not clips:raise ValueError('No completed clips yet.')
            if not shutil.which('ffmpeg'):raise RuntimeError('FFmpeg is missing from this runtime.')
            edit=project/'assembly';edit.mkdir(exist_ok=True)
            normalised=[]
            for i,source in enumerate(clips):
                target=edit/(str(i)+'.mp4')
                has_audio=subprocess.run(['ffprobe','-v','error','-select_streams','a','-show_entries','stream=index','-of','csv=p=0',str(source)],capture_output=True,text=True).stdout.strip()
                inputs=['-i',str(source)] if has_audio else ['-i',str(source),'-f','lavfi','-i','anullsrc=r=48000:cl=mono']
                result=subprocess.run(['ffmpeg','-y',*inputs,'-map','0:v:0','-map','0:a:0' if has_audio else '1:a:0','-shortest','-c:a','aac','-ar','48000','-ac','1','-vf','scale=640:360:force_original_aspect_ratio=decrease,pad=640:360:(ow-iw)/2:(oh-ih)/2,fps=24','-c:v','libx264','-preset','veryfast','-crf','20',str(target)],capture_output=True,text=True)
                if result.returncode:raise RuntimeError(result.stderr[-2000:])
                normalised.append(target)
            listing=edit/'concat.txt';listing.write_text(''.join("file '"+str(p)+"'\n" for p in normalised))
            target=project/'assembled_preview.mp4'
            result=subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i',str(listing),'-c','copy',str(target)],capture_output=True,text=True)
            if result.returncode:raise RuntimeError(result.stderr[-2000:])
            display(Video(str(target),embed=True));print('Assembled in generation order at 640×360. Voices retained; silent clips receive a silent audio track:',target)
        except Exception as exc:print('Assembly failed:',exc)
assemble.on_click(assemble_click)

# Lightweight speech: no GPU or voice model download.
voice_defaults={'CHEN':'en-US-JennyNeural','COLE':'en-GB-RyanNeural','VASIK':'en-GB-SoniaNeural','OSEI':'en-NG-EzinneNeural','ASANTE':'en-NG-AbeoNeural','TORRES':'en-US-GuyNeural','WEBB':'en-AU-WilliamNeural','NARRATOR':'en-GB-ThomasNeural'}
voice_map=w.Textarea(value=json.dumps(data.get('voices',voice_defaults),indent=2),description='Voices:',layout=w.Layout(width='95%',height='140px'))
speech=w.Textarea(value=data.get('last_speech','COLE: All personnel confirm suit integrity before we crack the seal.'),description='Speech:',placeholder='One SPEAKER: spoken line per line. NARRATOR is supported.',layout=w.Layout(width='95%',height='90px'))
voice_auto=w.Checkbox(value=False,description='Add speech after generation')
voice_rate=w.IntSlider(value=0,min=-20,max=20,description='Rate %:')
recording=w.Text(value='',description='Recording:',placeholder='Optional uploaded real voice recording replaces generated speech',layout=w.Layout(width='95%'))
voice_upload=w.Button(description='Upload real voice audio')
voice_apply=w.Button(description='Voice latest clip')
voice_preview=w.Button(description='Preview speech')
sound_choices=w.SelectMultiple(options=['Engine hum','Drill','Radio static','Alert beeps'],value=(),description='Sounds:')
sound_level=w.IntSlider(value=-24,min=-40,max=-10,description='SFX dB:')
def make_soundtrack(path,duration):
    import numpy as np,wave
    rate=24000;t=np.arange(int(duration*rate))/rate;rng=np.random.default_rng(42);audio=np.zeros(len(t))
    for effect in sound_choices.value:
        if effect=='Engine hum':audio+=0.45*np.sin(2*np.pi*55*t)+0.15*np.sin(2*np.pi*110*t)+0.06*rng.normal(size=len(t))
        elif effect=='Drill':audio+=(0.25*np.sin(2*np.pi*170*t)+0.18*rng.normal(size=len(t)))*(0.5+0.5*np.sin(2*np.pi*9*t)**2)
        elif effect=='Radio static':audio+=0.1*rng.normal(size=len(t))
        elif effect=='Alert beeps':audio+=0.25*np.sin(2*np.pi*880*t)*((t%2)<0.15)
    fade=np.minimum(1,t/0.15)*np.minimum(1,(duration-t)/0.15)
    audio=np.clip(audio*fade*10**(sound_level.value/20),-1,1)
    with wave.open(str(path),'wb') as output_wave:
        output_wave.setnchannels(1);output_wave.setsampwidth(2);output_wave.setframerate(rate);output_wave.writeframes((audio*32767).astype('<i2').tobytes())
    return path

def voice_settings():
    mapping=json.loads(voice_map.value)
    if not isinstance(mapping,dict) or any(not isinstance(v,str) or not v for v in mapping.values()):raise ValueError('Voice map must contain speaker names and voice IDs.')
    current=load();current.update(voices=mapping,last_speech=speech.value);save(current)
    return mapping

def media_seconds(path):
    r=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(path)],capture_output=True,text=True,check=True)
    return float(r.stdout.strip())

def make_speech(destination):
    import sys
    mapping=voice_settings();destination.mkdir(parents=True,exist_ok=True)
    if recording.value.strip():
        src=Path(recording.value.strip())
        if not src.is_file():raise FileNotFoundError(src)
        audio=destination/'recorded_voice.wav'
        r=subprocess.run(['ffmpeg','-y','-i',str(src),'-vn','-ar','48000','-ac','1',str(audio)],capture_output=True,text=True)
        if r.returncode:raise RuntimeError(r.stderr[-1500:])
        return audio,{'type':'human recording','source':str(src)}
    entries=[]
    for line in speech.value.splitlines():
        if not line.strip():continue
        who,sep,words=line.partition(':');who=who.strip().upper();words=words.strip()
        if not sep or not words or who not in mapping:raise ValueError('Use SPEAKER: words with a saved voice for every speaker.')
        entries.append((who,words))
    if not entries:raise ValueError('Enter dialogue/narration or upload a recording.')
    try:import edge_tts
    except ImportError:subprocess.run([sys.executable,'-m','pip','install','-q','edge-tts>=7,<8'],check=True)
    tracks=[]
    for index,(who,words) in enumerate(entries):
        textfile=destination/f'line_{index}.txt';textfile.write_text(words)
        mp3=destination/f'line_{index}.mp3';wav=destination/f'line_{index}.wav'
        command=[sys.executable,'-m','edge_tts','--file',str(textfile),'--voice',mapping[who],f'--rate={voice_rate.value:+d}%','--write-media',str(mp3)]
        r=subprocess.run(command,capture_output=True,text=True,timeout=120)
        if r.returncode or not mp3.is_file():raise RuntimeError('Free speech service failed; upload a real recording instead. '+r.stderr[-1500:])
        r=subprocess.run(['ffmpeg','-y','-i',str(mp3),'-ar','48000','-ac','1',str(wav)],capture_output=True,text=True)
        if r.returncode:raise RuntimeError(r.stderr[-1500:])
        tracks.append(wav)
    listing=destination/'speech_concat.txt';listing.write_text(''.join("file '"+str(p)+"'\n" for p in tracks))
    audio=destination/'speech.wav'
    r=subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i',str(listing),'-c:a','pcm_s16le',str(audio)],capture_output=True,text=True)
    if r.returncode:raise RuntimeError(r.stderr[-1500:])
    return audio,{'type':'synthetic neural speech','lines':entries,'voices':mapping,'rate':voice_rate.value}

def voice_clip(root):
    if speech.value.strip() or recording.value.strip():
        audio,metadata=make_speech(root/'voice')
    else:
        (root/'voice').mkdir(exist_ok=True)
        previous=sound_choices.value
        sound_choices.value=()
        try:audio=make_soundtrack(root/'voice'/'silence.wav',max(5.0,media_seconds(root/'ltx_cpu_test.mp4')))
        finally:sound_choices.value=previous
        metadata={'type':'sound only'}
    source=root/'ltx_cpu_test.mp4';duration=max(5.0,media_seconds(source),media_seconds(audio)+0.2)
    if sound_choices.value:
        effects=make_soundtrack(root/'voice'/'effects.wav',duration)
        mixed=root/'voice'/'mixed.wav'
        r=subprocess.run(['ffmpeg','-y','-i',str(audio),'-i',str(effects),'-filter_complex','[0:a]apad[a];[a][1:a]amix=inputs=2:duration=longest:normalize=0,alimiter=limit=0.95[out]','-map','[out]','-t',str(duration),str(mixed)],capture_output=True,text=True)
        if r.returncode:raise RuntimeError(r.stderr[-1500:])
        audio=mixed
    metadata.update(sound_effects=list(sound_choices.value),sound_db=sound_level.value)
    target=root/'voiced_video.mp4'
    r=subprocess.run(['ffmpeg','-y','-i',str(source),'-i',str(audio),'-map','0:v:0','-map','1:a:0','-vf',f'tpad=stop_mode=clone:stop_duration={duration}','-af','apad','-t',str(duration),'-c:v','libx264','-preset','veryfast','-crf','18','-c:a','aac','-b:a','192k',str(target)],capture_output=True,text=True)
    (root/'voice'/'merge.log').write_text(r.stderr)
    if r.returncode:raise RuntimeError(r.stderr[-1500:])
    metadata.update(duration=duration,lip_sync=False,video=str(target));(root/'voice'/'voice.json').write_text(json.dumps(metadata,indent=2))
    current=load()
    for clip in current['clips']:
        if Path(clip['folder'])==root:clip.update(voiced_video=str(target),voice=metadata)
    save(current)
    if duration>media_seconds(source)+0.1:print('Last frame held to fit the full voice line; no dialogue was cut.')
    print('Voice added. Mouth movements are NOT synchronized by this step.')
    return target

def voice_action(_,preview=False):
    with output:
        try:
            if preview:
                from IPython.display import Audio
                audio,_=make_speech(project/'voice_preview');display(Audio(filename=str(audio)));return
            clips=[c for c in load()['clips'] if c['status']=='complete']
            if not clips:raise ValueError('No completed video yet. Preview speech while video is queued.')
            display(Video(str(voice_clip(Path(clips[-1]['folder']))),embed=True))
        except Exception as exc:print('Voice failed:',exc)

def upload_voice(_):
    with output:
        from google.colab import files
        uploaded=files.upload()
        for name,content in uploaded.items():
            path=project/'recordings'/(uuid.uuid4().hex+Path(name).suffix);path.parent.mkdir(exist_ok=True);path.write_bytes(content)
            try:media_seconds(path)
            except Exception: path.unlink();print('Not readable audio:',name);continue
            recording.value=str(path);print('Recording ready:',path);break
voice_upload.on_click(upload_voice);voice_apply.on_click(voice_action);voice_preview.on_click(lambda b:voice_action(b,True))
voice_panel=w.VBox([w.HTML('<b>Voices and narration</b><p>Neural voices sound human but are synthetic. Upload an actual human recording to use a real voice. No GPU needed. Speech starts at the beginning, lines play sequentially. Audio merging does not provide lip sync.</p>'),speech,voice_map,voice_rate,recording,sound_choices,sound_level,w.HBox([voice_upload,voice_preview,voice_apply]),voice_auto])

def create(_):
    button.disabled=True
    with output:
        clear_output(wait=True)
        root=None;clip_id=None
        try:
            current=persist()
            if not prompt.value.strip():raise ValueError('Enter a scene prompt first.')
            python=Path('/content/ltx_cpu_env/bin/python')
            if not python.is_file():raise RuntimeError('Run the direct engine setup cell first.')
            if shutil.disk_usage('/content').free<20*1024**3:raise RuntimeError('Need 20 GB free disk.')
            if psutil.virtual_memory().available<10*1024**3:raise RuntimeError('Need 10 GB available RAM. Start a fresh CPU runtime.')
            names=[n.strip().upper() for n in cast.value.split(',') if n.strip()]
            missing=[n for n in names if not current['characters'].get(n)]
            if missing:raise ValueError('Add saved appearance/clothing descriptions for: '+', '.join(missing))
            if backend.value=='Direct Colab GPU':
                check=subprocess.run([str(python),'-c','import torch; print(torch.cuda.is_available())'],capture_output=True,text=True)
                if check.returncode or check.stdout.strip()!='True':raise RuntimeError('No Colab GPU available. Switch runtime to GPU when quota returns, or explicitly choose Experimental CPU.')
            ref=reference.value.strip()
            if ref and not Path(ref).is_file():raise FileNotFoundError('Reference image does not exist: '+ref)
            clip_id=datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6]
            root=project/'clips'/clip_id;root.mkdir(parents=True)
            identity=' '.join(n+': '+current['characters'][n] for n in names)
            effective=identity+' '+prompt.value.strip()+' '+current['world']+' '+current['style']
            width,height=(192,128) if size.value=='192×128' else (256,160)
            # This repository loads the base checkpoint, not a guidance-distilled model.
            guidance=5.0
            if steps.value<24:print('Low-step preview: reduced detail and motion quality are possible; this is not distilled inference.')
            print('Requested motion:',duration_choice.value,'seconds. Longer clips need more memory and processing time.')
            request={'prompt':prompt.value,'effective_prompt':effective,'cast':names,'seed':seed.value,'steps':steps.value,'reference_image':ref,'width':width,'height':height,'guidance':guidance,'max_tokens':192,'mode':'base checkpoint','frames':motion_frames(duration_choice.value),'fps':8,'requested_motion_seconds':duration_choice.value,'device':'cuda' if backend.value=='Direct Colab GPU' else 'cpu','model':'Lightricks/LTX-Video','backend':backend.value}
            if ref:
                original=Path(ref);snapshot=root/('reference'+original.suffix);shutil.copy2(original,snapshot);request['reference_image']=str(snapshot)
            (root/'request.json').write_text(json.dumps(request,indent=2))
            current['clips'].append({'id':clip_id,'status':'running','prompt':prompt.value,'folder':str(root),'request':request});save(current)
            if True:
                print('Generating',clip_id,'directly in Colab. CPU processing can take hours.')
                cache=project/'embedding_cache';cache.mkdir(exist_ok=True)
                fingerprint=hashlib.sha256(json.dumps({'prompt':effective,'model':request['model'],'guidance':guidance,'max_tokens':192,'schema':1},sort_keys=True).encode()).hexdigest()
                cached=cache/(fingerprint+'.pt')
                if cached.is_file():
                    shutil.copy2(cached,root/'embeddings.pt');print('Reusing saved prompt encoding — text encoder stage skipped.')
                stages=[('generate',generate)] if cached.is_file() else [('encode',encode),('generate',generate)]
                for name,source in stages:
                    script=Path('/content/ltx_cpu_'+name+'.py');script.write_text(source)
                    log=root/(name+'.log')
                    print('Starting',name,'— log:',log,flush=True)
                    with log.open('w') as stream:
                        proc=subprocess.Popen([str(python),'-u',str(script)],stdout=stream,stderr=subprocess.STDOUT,env={**os.environ,'USE_TF':'0','USE_FLAX':'0','TOKENIZERS_PARALLELISM':'false','LTX_RUN_DIR':str(root)})
                        started=time.time();last_report=0
                        try:
                            while proc.poll() is None:
                                elapsed=time.time()-started
                                if psutil.virtual_memory().available<650*1024**2:
                                    proc.terminate();proc.wait(timeout=30)
                                    raise RuntimeError('Stopped before exhausting RAM. This runtime is too small for this configuration.')
                                if elapsed>4*3600:
                                    proc.terminate();proc.wait(timeout=30);raise RuntimeError('Stopped after four hours. See saved log.')
                                if elapsed-last_report>30:
                                    print(name,'elapsed minutes:',round(elapsed/60,1),'RAM available GB:',round(psutil.virtual_memory().available/1024**3,2),flush=True)
                                    print('\n'.join(log.read_text(errors='replace').splitlines()[-3:]),flush=True);last_report=elapsed
                                time.sleep(3)
                        finally:
                            if proc.poll() is None:proc.kill();proc.wait()
                    print('\n'.join(log.read_text(errors='replace').splitlines()[-20:]))
                    if proc.returncode:raise RuntimeError(name+' failed; share the final log lines. No automatic larger-memory retry.')
                    if name=='encode':
                        part=cached.with_suffix('.part');shutil.copy2(root/'embeddings.pt',part);part.replace(cached)
            current=load()
            for clip in current['clips']:
                if clip['id']==clip_id:clip['status']='complete'
            save(current)
            shown=root/'ltx_cpu_test.mp4'
            if voice_auto.value:
                try:shown=voice_clip(root)
                except Exception as exc:print('Video saved, but voice failed:',exc)
            display(Video(str(shown),embed=True))
            print('Saved:',root/'ltx_cpu_test.mp4')
        except Exception as exc:
            if clip_id:
                current=load()
                for clip in current['clips']:
                    if clip['id']==clip_id:clip.update(status='failed',error=str(exc))
                save(current)
            print('Generation stopped:',exc)
        finally:button.disabled=False
savebutton.on_click(save_click);button.on_click(create);approve.on_click(approve_click);history.on_click(show_history)
display(w.VBox([w.HTML('<b>Book Video Studio — direct Colab generation + project memory</b><p>Project memory saves descriptions and references; it does not train the model or guarantee matching identities. Generation runs in your Colab runtime. No Spaces API, token, or ZeroGPU quota. GPU availability is controlled by Colab. CPU mode is experimental. Public model files still download from Hugging Face; no hosted inference is used.</p>'),backend,prompt,cast,characters,world,style,reference,upload,voice_panel,w.HBox([seed,steps]),w.HBox([size,duration_choice]),w.HBox([savebutton,button,approve,history]),w.HBox([assemble,backup]),output]))
